# Registros duplicados del dataset

Este notebook identifica filas exactamente repetidas en el dataset de delivery. La comparacion usa todas las columnas con datos y excluye unicamente la columna `Unnamed` vacia que aparece por la coma final del CSV. No se recortan ni normalizan valores.

Los numeros de fila reportados corresponden al archivo CSV e incluyen el encabezado como fila 1.

In [1]:
from pathlib import Path
import pandas as pd

ruta_dataset = Path('..') / 'data' / 'online food delivery dataset.csv'
df_original = pd.read_csv(ruta_dataset)
columnas_extra = [columna for columna in df_original.columns if str(columna).startswith('Unnamed')]
df = df_original.drop(columns=columnas_extra).copy()
columnas_comparacion = df.columns.tolist()

print(f'Filas analizadas: {len(df)}')
print(f'Columnas comparadas: {len(columnas_comparacion)}')
print(f'Columnas Unnamed excluidas: {columnas_extra}')

Filas analizadas: 388
Columnas comparadas: 13
Columnas Unnamed excluidas: ['Unnamed: 13']


In [3]:
mascara_duplicados = df.duplicated(subset=columnas_comparacion, keep=False)
duplicados = df.loc[mascara_duplicados].copy()
duplicados['_fila_csv'] = duplicados.index + 2

grupos_duplicados = (
    duplicados.groupby(columnas_comparacion, dropna=False, sort=False)
    .agg(veces=('_fila_csv', 'size'), filas_csv=('_fila_csv', list))
    .reset_index()
)
grupos_duplicados = grupos_duplicados.sort_values(
    'veces', ascending=False, kind='stable'
).reset_index(drop=True)

numero_grupos = len(grupos_duplicados)
filas_implicadas = int(grupos_duplicados['veces'].sum())
copias_extra = int((grupos_duplicados['veces'] - 1).sum())

print(f'Grupos de registros repetidos: {numero_grupos}')
print(f'Filas que pertenecen a esos grupos: {filas_implicadas}')
print(f'Copias adicionales a la primera aparicion: {copias_extra}')

Grupos de registros repetidos: 66
Filas que pertenecen a esos grupos: 169
Copias adicionales a la primera aparicion: 103


In [4]:
resumen_frecuencias = (
    grupos_duplicados.groupby('veces', as_index=False)
    .agg(
        grupos=('veces', 'size'),
        filas_implicadas=('veces', 'sum'),
        copias_extra=('veces', lambda frecuencia: (frecuencia - 1).sum()),
    )
.sort_values('veces', ascending=False)
.reset_index(drop=True)
)
display(resumen_frecuencias)

# La tabla contiene cada registro completo repetido, su frecuencia y las filas del CSV.
display(grupos_duplicados)

,veces,grupos,filas_implicadas,copias_extra
0,6,1,6,5
1,4,2,8,6
2,3,29,87,58
3,2,34,68,34


,Age,Gender,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,Customer Type,latitude,longitude,Pin code,Output,Feedback,veces,filas_csv
0,23,Female,Single,Student,No Income,Post Graduate,4,Frequent,13.0487,77.5923,560024,Yes,Positive,6,"[69, 70, 131, 153, 157, 277]"
1,22,Female,Single,Student,No Income,Post Graduate,5,Frequent,12.9850,77.5533,560010,Yes,Positive,4,"[12, 45, 310, 387]"
2,23,Female,Single,Student,No Income,Graduate,5,Frequent,13.0206,77.6479,560043,Yes,Positive,4,"[52, 98, 99, 115]"
3,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,Frequent,12.9850,77.5533,560010,Yes,Positive,3,"[6, 304, 381]"
4,27,Female,Married,Employee,More than 50000,Post Graduate,2,Regular,12.9299,77.6848,560103,Yes,Positive,3,"[7, 305, 382]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,26,Female,Married,Employee,Below Rs.10000,Graduate,6,Frequent,12.9859,77.6713,560093,No,Negative,2,"[248, 377]"
62,25,Male,Single,Self Employeed,More than 50000,School,3,Regular,12.9859,77.6713,560093,Yes,Positive,2,"[249, 378]"
63,31,Male,Prefer not to say,Employee,Below Rs.10000,Graduate,1,New,12.9866,77.4904,560091,No,Negative,2,"[250, 379]"
64,23,Female,Single,Employee,25001 to 50000,Post Graduate,2,Regular,12.9847,77.5491,560100,Yes,Positive,2,"[251, 380]"
